# GRUPO:
GABRIEL GUIMARÃES DE OLIVEIRA - RM 567835

PEDRO PAULO FERREIRA AGNELO D'ANGELO - RM 567564

# Análise da Demanda por Bicicletas Compartilhadas

## Dataset

O dataset utilizado é o **Bike Sharing Dataset**, que contém informações sobre a utilização de um sistema de compartilhamento de bicicletas em Washington, D.C., durante os anos de 2011 e 2012.

Entre as variáveis disponíveis estão horário, temperatura, umidade, velocidade do vento, estação do ano, condições climáticas, tipo de dia e quantidade de bicicletas alugadas.

O arquivo utilizado nesta análise é o `hour.csv`, que contém os registros organizados por hora.

**Fonte:** Kaggle  
**URL:** https://www.kaggle.com/datasets/sriramm2010/uci-bike-sharing-data

### Perguntas investigativas

1. **Em quais horários ocorre a maior demanda por bicicletas?**
2. **A demanda por bicicletas apresenta diferenças entre dias úteis e dias não úteis?**
3. **Existe relação entre a temperatura e a quantidade de bicicletas alugadas?**
4. **Como as diferentes condições climáticas estão relacionadas à demanda por bicicletas?**

As perguntas foram escolhidas para analisar diferentes aspectos do comportamento da demanda e permitir a utilização de diferentes tipos de visualização.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv("hour.csv")

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [ ]:
print("Número de linhas:", df.shape[0])
print("Número de colunas:", df.shape[1])

df.info()

Número de linhas: 17379
Número de colunas: 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


In [ ]:
df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
instant,0
dteday,0
season,0
yr,0
mnth,0
hr,0
holiday,0
weekday,0
workingday,0
weathersit,0


In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"])

df["ano"] = df["yr"].map({
    0: 2011,
    1: 2012
})

df["tipo_dia"] = df["workingday"].map({
    0: "Não útil",
    1: "Dia útil"
})

df["clima"] = df["weathersit"].map({
    1: "Céu limpo / parcialmente nublado",
    2: "Nublado / neblina",
    3: "Chuva ou neve leve",
    4: "Chuva forte"
})

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,ano,tipo_dia,clima
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011,Não útil,Céu limpo / parcialmente nublado
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011,Não útil,Céu limpo / parcialmente nublado
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011,Não útil,Céu limpo / parcialmente nublado
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011,Não útil,Céu limpo / parcialmente nublado
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011,Não útil,Céu limpo / parcialmente nublado


## Preparação dos dados

A variável `dteday` foi convertida para o formato de data para facilitar análises temporais.

As variáveis `workingday` e `weathersit`, originalmente representadas por códigos numéricos, foram transformadas em categorias com descrições mais compreensíveis. Isso facilita a interpretação dos gráficos.

A variável `cnt` representa a quantidade total de bicicletas alugadas em cada registro horário e será utilizada como principal medida de demanda nesta análise.

As variáveis `temp`, `hum` e `windspeed` estão representadas em valores normalizados no dataset original. Portanto, os gráficos que utilizam essas variáveis mantêm a escala fornecida pelo dataset.

In [ ]:
df[["temp", "hum", "windspeed", "casual", "registered", "cnt"]].describe()

,temp,hum,windspeed,casual,registered,cnt
count,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000
mean,0.496987,0.627229,0.190098,35.676218,153.786869,189.463088
std,0.192556,0.192930,0.122340,49.305030,151.357286,181.387599
min,0.020000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.340000,0.480000,0.104500,4.000000,34.000000,40.000000
50%,0.500000,0.630000,0.194000,17.000000,115.000000,142.000000
75%,0.660000,0.780000,0.253700,48.000000,220.000000,281.000000
max,1.000000,1.000000,0.850700,367.000000,886.000000,977.000000


In [ ]:
media_hora = (
    df.groupby("hr", as_index=False)["cnt"]
    .mean()
)

media_hora.head()

,hr,cnt
0,0,53.898072
1,1,33.375691
2,2,22.869930
3,3,11.727403
4,4,6.352941


In [ ]:
fig = px.line(
    media_hora,
    x="hr",
    y="cnt"
)

fig.show()

## Análise do gráfico padrão — Gráfico 1

O gráfico padrão permite observar a variação da demanda média ao longo das horas do dia. Entretanto, sua capacidade de comunicação é limitada, pois não apresenta um título explicativo, os nomes dos eixos não estão contextualizados e os pontos de maior interesse não recebem destaque.

Por esse motivo, serão realizadas alterações no título, nos rótulos dos eixos, nos marcadores e na apresentação geral do gráfico.


In [ ]:
fig = px.line(
    media_hora,
    x="hr",
    y="cnt",
    markers=True,
    title="Demanda média de bicicletas ao longo do dia",
    labels={
        "hr": "Hora do dia",
        "cnt": "Média de bicicletas alugadas"
    }
)

fig.update_traces(
    line_width=3,
    marker_size=7
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis=dict(
        dtick=2
    ),
    hovermode="x unified"
)

fig.show()

## Justificativa da customização — Gráfico 1

As alterações foram feitas com o objetivo de melhorar a interpretação dos dados, e não apenas para modificar a aparência do gráfico.

Os marcadores foram adicionados para facilitar a identificação dos valores correspondentes a cada hora. A espessura da linha foi aumentada para melhorar sua visualização. O título explica diretamente o fenômeno apresentado e os rótulos dos eixos tornam as variáveis compreensíveis sem depender do conhecimento prévio das abreviações do dataset.

O espaçamento dos rótulos do eixo X foi ajustado para apresentar as horas em intervalos de duas unidades, evitando excesso de informação visual e facilitando a identificação dos horários.

## Interpretação — Gráfico 1

A demanda por bicicletas não se distribui igualmente ao longo do dia. Existem horários em que a quantidade média de aluguéis aumenta significativamente, indicando períodos de maior utilização do sistema.

Esse comportamento sugere que o horário possui relação importante com a utilização das bicicletas, possivelmente devido aos padrões de deslocamento das pessoas.

Dessa forma, o gráfico responde à primeira pergunta ao permitir identificar visualmente os períodos de maior e menor demanda.


In [ ]:
fig = px.box(
    df,
    x="tipo_dia",
    y="cnt"
)

fig.show()

## Análise do gráfico padrão — Gráfico 2

O box plot padrão permite comparar a distribuição da quantidade de bicicletas alugadas entre dias úteis e dias não úteis.

Entretanto, a versão padrão apresenta pouca contextualização sobre o significado das variáveis e não possui um título que indique claramente a comparação realizada.

A customização será utilizada para melhorar a comunicação da comparação entre os dois grupos.


In [ ]:
fig = px.box(
    df,
    x="tipo_dia",
    y="cnt",
    points=False,
    title="Distribuição da demanda em dias úteis e não úteis",
    labels={
        "tipo_dia": "Tipo de dia",
        "cnt": "Bicicletas alugadas por hora"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Justificativa da customização — Gráfico 2

Os pontos individuais foram removidos na versão customizada para evitar excesso de informação visual, já que o dataset possui milhares de registros.

O título e os rótulos dos eixos foram adicionados para deixar explícita a comparação realizada. O box plot é adequado para essa pergunta porque permite observar não apenas a média, mas também a mediana, a dispersão e possíveis valores extremos de cada grupo.


## Interpretação — Gráfico 2

A distribuição da quantidade de bicicletas alugadas apresenta diferenças entre dias úteis e dias não úteis.

O box plot permite observar essas diferenças considerando a distribuição completa dos dados, e não apenas um valor médio. A comparação das medianas e da dispersão ajuda a compreender como o comportamento da demanda varia de acordo com o tipo de dia.

Assim, a visualização permite avaliar se a rotina dos dias úteis está associada a um padrão diferente de utilização do sistema.


In [ ]:
amostra = df.sample(3000, random_state=42)

fig = px.scatter(
    amostra,
    x="temp",
    y="cnt"
)

fig.show()

## Análise do gráfico padrão — Gráfico 3

O gráfico de dispersão permite observar a relação entre temperatura e quantidade de bicicletas alugadas.

Porém, devido à grande quantidade de observações, muitos pontos ficam sobrepostos, dificultando a identificação da tendência geral dos dados.

A customização irá utilizar transparência nos pontos e uma linha de tendência para facilitar a identificação da relação entre as variáveis.


In [ ]:
fig = px.scatter(
    amostra,
    x="temp",
    y="cnt",
    opacity=0.45,
    trendline="ols",
    title="Relação entre temperatura e demanda por bicicletas",
    labels={
        "temp": "Temperatura normalizada",
        "cnt": "Bicicletas alugadas por hora"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Justificativa da customização — Gráfico 3

A transparência dos pontos foi utilizada para reduzir o problema de sobreposição entre observações.

Uma amostra aleatória de 3.000 registros foi utilizada para tornar a visualização mais legível e reduzir a sobreposição entre os pontos. A linha de tendência foi adicionada para facilitar a identificação da direção da relação entre temperatura e demanda.

Essas alterações possuem finalidade analítica, pois ajudam a identificar padrões que seriam mais difíceis de observar no gráfico padrão.


## Interpretação — Gráfico 3

O gráfico indica uma tendência positiva entre a temperatura e a demanda por bicicletas: conforme a temperatura aumenta, a quantidade de bicicletas alugadas tende a aumentar.

Apesar dessa tendência, existe uma dispersão considerável dos pontos, indicando que a temperatura não explica sozinha a variação da demanda. Outros fatores presentes no dataset também podem estar relacionados à utilização das bicicletas.

Portanto, os dados indicam uma associação entre temperatura e demanda, mas não permitem afirmar que a temperatura seja, isoladamente, a causa do aumento dos aluguéis.

In [ ]:
media_clima = (
    df.groupby("clima", as_index=False)["cnt"]
    .mean()
    .sort_values("cnt", ascending=False)
)

media_clima

,clima,cnt
2,Céu limpo / parcialmente nublado,204.869272
3,Nublado / neblina,175.165493
1,Chuva ou neve leve,111.579281
0,Chuva forte,74.333333


In [ ]:
fig = px.bar(
    media_clima,
    x="clima",
    y="cnt"
)

fig.show()

## Análise do gráfico padrão — Gráfico 4

O gráfico de barras padrão permite comparar a demanda média entre as diferentes condições climáticas.

Entretanto, a visualização ainda necessita de informações contextuais para que o leitor compreenda imediatamente o que está sendo comparado. Os rótulos dos eixos e o título serão modificados para tornar a mensagem principal mais clara.


In [ ]:
fig = px.bar(
    media_clima,
    x="clima",
    y="cnt",
    text_auto=".0f",
    title="Demanda média de bicicletas por condição climática",
    labels={
        "clima": "Condição climática",
        "cnt": "Média de bicicletas alugadas"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_tickangle=-20
)

fig.show()

## Justificativa da customização — Gráfico 4

Os valores foram apresentados diretamente sobre as barras para permitir uma comparação mais precisa entre as categorias.

O título e os rótulos dos eixos foram utilizados para contextualizar a informação apresentada. A rotação dos nomes das categorias foi aplicada para evitar sobreposição dos textos e melhorar a legibilidade.

As alterações têm como objetivo facilitar a comparação entre as condições climáticas e não apenas modificar a estética do gráfico.


## Interpretação — Gráfico 4

A demanda média apresenta diferenças de acordo com a condição climática.

As condições mais favoráveis apresentam maior utilização do sistema, enquanto condições climáticas mais desfavoráveis estão associadas a alterações na demanda.

Esse resultado indica que o clima é um dos fatores relacionados ao comportamento dos usuários, podendo influenciar a decisão de utilizar ou não o sistema de bicicletas compartilhadas.


# Conclusão

A análise do Bike Sharing Dataset permitiu investigar diferentes fatores relacionados à utilização de bicicletas compartilhadas.

As quatro visualizações apresentaram perspectivas diferentes sobre o comportamento da demanda. A análise por horário permitiu identificar períodos de maior e menor utilização. O box plot possibilitou comparar a distribuição da demanda entre dias úteis e não úteis. O gráfico de dispersão permitiu investigar a relação entre temperatura e quantidade de bicicletas alugadas, enquanto o gráfico de barras possibilitou comparar a demanda média entre diferentes condições climáticas.

A utilização de diferentes tipos de gráficos foi importante porque cada pergunta exigia uma abordagem visual específica. Além disso, a comparação entre as versões padrão e customizadas demonstrou que a personalização dos gráficos pode melhorar significativamente a comunicação e a interpretação dos dados.

Dessa forma, as visualizações foram utilizadas não apenas para apresentar informações, mas como ferramentas de investigação para identificar padrões e relações presentes no dataset.
